# 05 — Постобработка OCR и привязка страниц к FB2

## Цель

Нормализовать OCR-текст из выбранного эксперимента, сначала найти соответствующий фрагмент FB2 для разворота, затем уточнить позиции левой и правой страниц и сформировать проверяемый отчёт о качестве.

Постобработка намеренно не исправляет слова по словарю и не подставляет текст из FB2: она устраняет только безопасные артефакты OCR-разметки — Unicode-варианты, пробелы и переносы с дефисом. FB2 используется только для matching и диагностики.

Входы:

- `outputs/besy/processed_images/experiment_03_lower_box_threshold/`;
- `outputs/besy/run_01/normalized_text.pkl` из ноутбука 01.

Результаты сохраняются в `outputs/besy/ocr_matching/<эксперимент>/`.

In [1]:
from __future__ import annotations

import csv
from bisect import bisect_right
import json
import os
import pickle
import re
import unicodedata
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

PROJECT_ROOT = Path(os.environ.get(
    'SPARK_ROOT',
    '/home/wsl_user/my_projects/SPARK — Synchronized Print-Audio Reading Kit',
))
OCR_EXPERIMENT = 'experiment_03_lower_box_threshold'
OCR_DIR = PROJECT_ROOT / 'outputs/besy/processed_images' / OCR_EXPERIMENT
TEXT_DATA_PATH = PROJECT_ROOT / 'outputs/besy/run_01/normalized_text.pkl'
OUTPUT_DIR = PROJECT_ROOT / 'outputs/besy/ocr_matching' / OCR_EXPERIMENT
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

assert OCR_DIR.is_dir(), f'Не найдены OCR-результаты: {OCR_DIR}'
assert TEXT_DATA_PATH.is_file(), (
    f'Не найден нормализованный FB2: {TEXT_DATA_PATH}. Сначала выполните 01_text_normalization.ipynb.'
)
print(f'OCR: {OCR_DIR}')
print(f'FB2 index: {TEXT_DATA_PATH}')
print(f'Результаты: {OUTPUT_DIR}')


OCR: /home/wsl_user/my_projects/SPARK — Synchronized Print-Audio Reading Kit/outputs/besy/processed_images/experiment_03_lower_box_threshold
FB2 index: /home/wsl_user/my_projects/SPARK — Synchronized Print-Audio Reading Kit/outputs/besy/run_01/normalized_text.pkl
Результаты: /home/wsl_user/my_projects/SPARK — Synchronized Print-Audio Reading Kit/outputs/besy/ocr_matching/experiment_03_lower_box_threshold


In [2]:
def to_builtin(value):
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, dict):
        return {key: to_builtin(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_builtin(item) for item in value]
    return value


def write_json(path: Path, payload: dict | list) -> None:
    path.write_text(
        json.dumps(to_builtin(payload), ensure_ascii=False, indent=2),
        encoding='utf-8',
    )


def postprocess_ocr_text(text: str) -> str:
    """Исправить только безопасные артефакты строчного OCR."""
    text = unicodedata.normalize('NFKC', text)
    # Перенос вида «сло-\nво» — артефакт разбиения строки, а не часть слова.
    text = re.sub(r'(?<=[А-Яа-яЁё])-\s*\n\s*(?=[А-Яа-яЁё])', '', text)
    text = re.sub(r'[\t\r\n ]+', ' ', text)
    text = re.sub(r'\s+([,.;:!?])', r'\1', text)
    text = re.sub(r'([,.;:!?])(?!\s|$)', r'\1 ', text)
    return re.sub(r'\s+', ' ', text).strip()


def normalize_for_match(text: str) -> str:
    """Подготовить текст к устойчивому символьному сопоставлению."""
    text = postprocess_ocr_text(text).lower().replace('ё', 'е')
    text = re.sub(r'[^а-я0-9\s]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()


def spread_numbers(name: str) -> tuple[int, int]:
    match = re.fullmatch(r'(\d+)-(\d+)', name)
    if not match:
        raise ValueError(f'Ожидалось имя разворота вида 10-11, получено: {name}')
    return int(match.group(1)), int(match.group(2))


@dataclass
class OCRPage:
    page_number: int
    spread: str
    side: str
    raw_text: str
    postprocessed_text: str
    match_text: str


def load_ocr_pages(ocr_dir: Path) -> list[OCRPage]:
    pages = []
    spread_dirs = sorted(
        (path for path in ocr_dir.iterdir() if path.is_dir()),
        key=lambda path: spread_numbers(path.name),
    )
    for spread_dir in spread_dirs:
        left_page, right_page = spread_numbers(spread_dir.name)
        for side, page_number in [('left', left_page), ('right', right_page)]:
            text_path = spread_dir / f'{side}.txt'
            raw_text = text_path.read_text(encoding='utf-8')
            postprocessed_text = postprocess_ocr_text(raw_text)
            pages.append(OCRPage(
                page_number=page_number,
                spread=spread_dir.name,
                side=side,
                raw_text=raw_text,
                postprocessed_text=postprocessed_text,
                match_text=normalize_for_match(raw_text),
            ))
    return sorted(pages, key=lambda page: page.page_number)


In [3]:
ocr_pages = load_ocr_pages(OCR_DIR)
assert ocr_pages, 'Не найдены страницы OCR.'
assert all(page.match_text for page in ocr_pages), 'Одна из страниц пуста после нормализации.'

with TEXT_DATA_PATH.open('rb') as file:
    fb2_data = pickle.load(file)

# Сопоставляем именно с тем же логическим потоком, что и аудио в 03c.
# Его координаты можно напрямую использовать в двунаправленном поиске.
book_text = fb2_data['matching_text']
matching_segments = fb2_data['matching_segments']
matching_segment_starts = [segment['match_char_start'] for segment in matching_segments]
assert book_text and matching_segments, 'Логический поток FB2 пуст. Перезапустите 01_text_normalization.ipynb.'

write_json(OUTPUT_DIR / 'ocr_pages_postprocessed.json', [asdict(page) for page in ocr_pages])
with (OUTPUT_DIR / 'ocr_pages_postprocessed.txt').open('w', encoding='utf-8') as file:
    for page in ocr_pages:
        file.write(f'## {page.page_number} ({page.spread}, {page.side})\n')
        file.write(page.postprocessed_text + '\n\n')

print(f'Страниц OCR: {len(ocr_pages)}')
print(f'Длина логического FB2 для matching: {len(book_text):,} символов')
for page in ocr_pages:
    print(f'{page.page_number}: {len(page.match_text):,} символов')


Страниц OCR: 14
Длина логического FB2 для matching: 1,881,557 символов
10: 2,193 символов
11: 1,917 символов
96: 1,625 символов
97: 1,838 символов
182: 2,074 символов
183: 1,883 символов
356: 1,681 символов
357: 1,731 символов
528: 2,113 символов
529: 2,069 символов
614: 2,014 символов
615: 1,920 символов
698: 1,685 символов
699: 1,576 символов


## Matching

Поиск идёт в два этапа и остаётся словным TF-IDF. Сначала весь разворот ищется в крупных окнах FB2: обе страницы дают один общий контекст и устраняют случайные совпадения. Затем левая и правая страницы уточняются в этом контексте малыми окнами; выбирается только пара в порядке чтения.

Уверенность разворота основана на трёх признаках: TF-IDF общего контекста, разнице с лучшим непересекающимся регионом FB2 и соблюдении порядка «левая → правая». Последовательность позиций между страницами выводится отдельно как диагностика. Это оценка надёжности matching, а не доказательство дословной точности OCR.

In [4]:
# Первый этап ищет разворот целиком: так соседние страницы подтверждают друг друга.
CONTEXT_WINDOW_SIZE = 10_000
CONTEXT_WINDOW_STEP = 1_500
# Второй этап уточняет каждую страницу только внутри найденного контекста.
LOCAL_WINDOW_SIZE = 800
LOCAL_WINDOW_STEP = 100
TOP_WINDOWS_PER_QUERY = 80
MAX_PAGE_GAP = 9_000


def resolve_reading_position(match_char_pos: int) -> dict:
    """Перевести позицию логического matching-потока в общую координату чтения."""
    index = bisect_right(matching_segment_starts, match_char_pos) - 1
    index = max(0, min(index, len(matching_segments) - 1))
    segment = matching_segments[index]
    if segment['kind'] == 'note':
        reading_position = segment['reading_position']
    else:
        offset = max(0, match_char_pos - segment['match_char_start'])
        reading_position = segment['reading_position'] + offset
    return {'char_start': reading_position, 'section': segment['source']}


def build_windows(text: str, window_size: int, step: int, offset: int = 0) -> list[dict]:
    if len(text) <= window_size:
        return [{'char_start': offset, 'char_end': offset + len(text), 'text': text}]

    windows = []
    for start in range(0, len(text) - window_size + 1, step):
        windows.append({
            'char_start': offset + start,
            'char_end': offset + start + window_size,
            'text': text[start:start + window_size],
        })
    if windows and windows[-1]['char_end'] < offset + len(text):
        windows.append({
            'char_start': offset + len(text) - window_size,
            'char_end': offset + len(text),
            'text': text[-window_size:],
        })
    return windows


def cluster_regions(scores: np.ndarray, windows: list[dict]) -> list[dict]:
    """Склеить перекрывающиеся окна в независимые кандидаты-регионы."""
    candidate_count = min(TOP_WINDOWS_PER_QUERY, len(windows))
    indexes = np.argsort(scores)[-candidate_count:]
    indexes = sorted((int(index) for index in indexes), key=lambda index: windows[index]['char_start'])
    regions = []
    for index in indexes:
        window = windows[index]
        if not regions or window['char_start'] > regions[-1]['char_end']:
            regions.append({'indexes': [index], 'char_start': window['char_start'], 'char_end': window['char_end']})
        else:
            region = regions[-1]
            region['indexes'].append(index)
            region['char_end'] = max(region['char_end'], window['char_end'])
    for region in regions:
        best_index = max(region['indexes'], key=lambda index: scores[index])
        region['best_index'] = best_index
        region['score'] = float(scores[best_index])
    return sorted(regions, key=lambda region: region['score'], reverse=True)


def top_indexes(scores: np.ndarray, count: int = 5) -> list[int]:
    return [int(index) for index in np.argsort(scores)[-count:][::-1]]


def choose_page_pair(left_scores, right_scores, local_windows):
    left_indexes = top_indexes(left_scores, TOP_WINDOWS_PER_QUERY)
    right_indexes = top_indexes(right_scores, TOP_WINDOWS_PER_QUERY)
    ordered_pairs = [
        (left_scores[left_index] + right_scores[right_index], left_index, right_index)
        for left_index in left_indexes
        for right_index in right_indexes
        if 0 <= local_windows[right_index]['char_start'] - local_windows[left_index]['char_start'] <= MAX_PAGE_GAP
    ]
    if ordered_pairs:
        _, left_index, right_index = max(ordered_pairs, key=lambda item: item[0])
        return int(left_index), int(right_index), True
    return left_indexes[0], right_indexes[0], False


pages_by_spread = {}
for page in ocr_pages:
    pages_by_spread.setdefault(page.spread, {})[page.side] = page

spreads = []
for spread_name, spread_pages in pages_by_spread.items():
    assert set(spread_pages) == {'left', 'right'}, f'Неполный разворот: {spread_name}'
    spreads.append({
        'spread': spread_name,
        'left': spread_pages['left'],
        'right': spread_pages['right'],
        'match_text': f"{spread_pages['left'].match_text} {spread_pages['right'].match_text}",
    })

context_windows = build_windows(book_text, CONTEXT_WINDOW_SIZE, CONTEXT_WINDOW_STEP)
context_texts = [window['text'] for window in context_windows]
spread_texts = [spread['match_text'] for spread in spreads]
context_vectorizer = TfidfVectorizer(analyzer='word', max_features=50_000)
context_matrix = context_vectorizer.fit_transform(context_texts + spread_texts)
context_similarities = cosine_similarity(
    context_matrix[len(context_windows):],
    context_matrix[:len(context_windows)],
)

spread_matches = []
matches = []
for spread_index, spread in enumerate(spreads):
    context_scores = context_similarities[spread_index]
    regions = cluster_regions(context_scores, context_windows)
    best_region = regions[0]
    second_region_score = regions[1]['score'] if len(regions) > 1 else 0.0
    region_margin = best_region['score'] - second_region_score

    local_start = max(0, best_region['char_start'] - CONTEXT_WINDOW_STEP)
    local_end = min(len(book_text), best_region['char_end'] + CONTEXT_WINDOW_STEP)
    local_windows = build_windows(
        book_text[local_start:local_end], LOCAL_WINDOW_SIZE, LOCAL_WINDOW_STEP, offset=local_start
    )
    local_texts = [window['text'] for window in local_windows]
    local_vectorizer = TfidfVectorizer(analyzer='word', max_features=50_000)
    local_matrix = local_vectorizer.fit_transform(
        local_texts + [spread['left'].match_text, spread['right'].match_text]
    )
    local_scores = cosine_similarity(local_matrix[-2:], local_matrix[:-2])
    left_index, right_index, page_order_valid = choose_page_pair(
        local_scores[0], local_scores[1], local_windows
    )

    spread_match = {
        'spread': spread['spread'],
        'context_char_start': best_region['char_start'],
        'context_char_end': best_region['char_end'],
        'context_tfidf_score': best_region['score'],
        'independent_region_margin': region_margin,
        'alternative_region_score': second_region_score,
        'page_order_valid': page_order_valid,
        'candidate_regions': [
            {
                'book_char_start': region['char_start'],
                'book_char_end': region['char_end'],
                'tfidf_score': region['score'],
            }
            for region in regions[:3]
        ],
    }
    spread_matches.append(spread_match)

    for page, window_index, scores in (
        (spread['left'], left_index, local_scores[0]),
        (spread['right'], right_index, local_scores[1]),
    ):
        window = local_windows[window_index]
        candidate_indexes = top_indexes(scores)
        reading_location = resolve_reading_position(window['char_start'])
        matches.append({
            'page_number': page.page_number,
            'spread': page.spread,
            'side': page.side,
            # book_char_* — координаты общей шкалы чтения, одинаковой с audio_map.
            'book_char_start': reading_location['char_start'],
            'book_char_end': reading_location['char_start'] + LOCAL_WINDOW_SIZE,
            'matching_char_start': window['char_start'],
            'matching_char_end': window['char_end'],
            'section': reading_location['section'],
            'local_tfidf_score': float(scores[window_index]),
            'context_tfidf_score': best_region['score'],
            'independent_region_margin': region_margin,
            'page_order_valid': page_order_valid,
            'top_candidates': [
                {
                    'book_char_start': local_windows[index]['char_start'],
                    'tfidf_score': float(scores[index]),
                }
                for index in candidate_indexes
            ],
            'ocr_preview': page.postprocessed_text[:250],
            'fb2_preview': window['text'][:250],
        })

matches.sort(key=lambda match: match['page_number'])
print(f'Сопоставлено разворотов: {len(spread_matches)}, страниц: {len(matches)}')


Сопоставлено разворотов: 7, страниц: 14


In [5]:
def confidence_label(spread_match: dict) -> str:
    """Оценить однозначность региона, а не различие соседних окон."""
    if spread_match['context_tfidf_score'] < 0.10:
        return 'weak'
    if spread_match['page_order_valid'] and spread_match['independent_region_margin'] >= 0.05:
        return 'reliable'
    if spread_match['page_order_valid'] and spread_match['independent_region_margin'] >= 0.015:
        return 'plausible'
    return 'ambiguous'


for spread_match in spread_matches:
    spread_match['confidence'] = confidence_label(spread_match)

confidence_by_spread = {spread_match['spread']: spread_match['confidence'] for spread_match in spread_matches}
previous_match = None
monotonic_violations = []
for match in matches:
    match['confidence'] = confidence_by_spread[match['spread']]
    match['monotonic'] = previous_match is None or match['book_char_start'] > previous_match['book_char_start']
    if not match['monotonic']:
        monotonic_violations.append({
            'page_number': match['page_number'],
            'book_char_start': match['book_char_start'],
            'previous_page_number': previous_match['page_number'],
            'previous_book_char_start': previous_match['book_char_start'],
        })
    previous_match = match

confidence_counts = {
    label: sum(spread_match['confidence'] == label for spread_match in spread_matches)
    for label in ('reliable', 'plausible', 'ambiguous', 'weak')
}
summary = {
    'ocr_experiment': OCR_EXPERIMENT,
    'matching_method': 'two_stage_word_tfidf',
    'context_window_size': CONTEXT_WINDOW_SIZE,
    'context_window_step': CONTEXT_WINDOW_STEP,
    'local_window_size': LOCAL_WINDOW_SIZE,
    'local_window_step': LOCAL_WINDOW_STEP,
    'pages_total': len(matches),
    'spreads_total': len(spread_matches),
    'confidence_counts_by_spread': confidence_counts,
    'mean_context_tfidf_score': float(np.mean([item['context_tfidf_score'] for item in spread_matches])),
    'mean_independent_region_margin': float(np.mean([item['independent_region_margin'] for item in spread_matches])),
    'page_order_valid_spreads': sum(item['page_order_valid'] for item in spread_matches),
    'monotonic_violations': monotonic_violations,
}
write_json(OUTPUT_DIR / 'ocr_spread_matches.json', spread_matches)
write_json(OUTPUT_DIR / 'ocr_page_matches.json', matches)
write_json(OUTPUT_DIR / 'quality_summary.json', summary)

csv_columns = [
    'page_number', 'spread', 'side', 'book_char_start', 'book_char_end',
    'matching_char_start', 'matching_char_end', 'section',
    'confidence', 'local_tfidf_score', 'context_tfidf_score',
    'independent_region_margin', 'page_order_valid', 'monotonic',
]
with (OUTPUT_DIR / 'ocr_page_matches.csv').open('w', encoding='utf-8', newline='') as file:
    writer = csv.DictWriter(file, fieldnames=csv_columns)
    writer.writeheader()
    writer.writerows([{key: match[key] for key in csv_columns} for match in matches])

report = [
    '# OCR → FB2 matching quality',
    '',
    f'- OCR experiment: `{OCR_EXPERIMENT}`',
    f'- Pages matched: {len(matches)}; spreads: {len(spread_matches)}',
    f'- Reliable / plausible / ambiguous / weak spreads: ' +
    f"{confidence_counts['reliable']} / {confidence_counts['plausible']} / " +
    f"{confidence_counts['ambiguous']} / {confidence_counts['weak']}",
    f'- Mean context word TF-IDF: {summary["mean_context_tfidf_score"]:.3f}',
    f'- Mean margin to the next independent region: {summary["mean_independent_region_margin"]:.3f}',
    f'- Spreads with left → right page order: {summary["page_order_valid_spreads"]}/{len(spread_matches)}',
    f'- Sequential-position violations: {len(monotonic_violations)}',
    '',
    'A small margin is meaningful here only when the second score belongs to another, non-overlapping FB2 region.',
    '',
    '| Spread | Confidence | Context TF-IDF | Independent margin | Left → right |',
    '|---|---|---:|---:|---|',
]
for spread_match in spread_matches:
    report.append(
        f"| {spread_match['spread']} | {spread_match['confidence']} | "
        f"{spread_match['context_tfidf_score']:.3f} | "
        f"{spread_match['independent_region_margin']:.3f} | "
        f"{spread_match['page_order_valid']} |"
    )

for match in matches:
    if match['confidence'] in ('ambiguous', 'weak') or not match['monotonic']:
        report.extend([
            '',
            f"## Page {match['page_number']} — {match['confidence']}",
            '',
            f"OCR: {match['ocr_preview']}",
            '',
            f"FB2: {match['fb2_preview']}",
        ])

(OUTPUT_DIR / 'quality_report.md').write_text('\n'.join(report) + '\n', encoding='utf-8')

print(json.dumps(summary, ensure_ascii=False, indent=2))
for spread_match in spread_matches:
    print(
        f"{spread_match['spread']}: {spread_match['confidence']}, "
        f"score={spread_match['context_tfidf_score']:.3f}, "
        f"margin={spread_match['independent_region_margin']:.3f}"
    )


{
  "ocr_experiment": "experiment_03_lower_box_threshold",
  "matching_method": "two_stage_word_tfidf",
  "context_window_size": 10000,
  "context_window_step": 1500,
  "local_window_size": 800,
  "local_window_step": 100,
  "pages_total": 14,
  "spreads_total": 7,
  "confidence_counts_by_spread": {
    "reliable": 7,
    "plausible": 0,
    "ambiguous": 0,
    "weak": 0
  },
  "mean_context_tfidf_score": 0.7016234599020142,
  "mean_independent_region_margin": 0.49172378885649765,
  "page_order_valid_spreads": 7,
  "monotonic_violations": []
}
10-11: reliable, score=0.685, margin=0.475
96-97: reliable, score=0.668, margin=0.461
182-183: reliable, score=0.720, margin=0.432
356-357: reliable, score=0.720, margin=0.553
528-529: reliable, score=0.715, margin=0.499
614-615: reliable, score=0.587, margin=0.376
698-699: reliable, score=0.817, margin=0.646
